# 🏛️ Political Accountability Layer

**The Data Vigilante | Congressional Oversight Extension**

Maps the legislative control over UI/SUI policy to member metadata, committee assignments, and constituent demographics.

## Data Sources
- **Congress.gov API v3**: Member metadata, committee assignments (RATE LIMITED - cached from public directory)
- **Census ACS 2022**: Median household income by congressional district (✅ verified via api.census.gov)
- **OpenSecrets**: Financial disclosures (❌ Cloudflare blocked - documented as future integration)

## Honest Limitations
- Member data constructed from Congress.gov public directory (verified against official listings)
- Committee assignments subject to change (119th Congress, 2025–2027)
- OpenSecrets integration requires API key registration (blocked by Cloudflare challenge)
- When Congress.gov API rate limits reset, re-run `political_layer_builder.py` for live verification

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150

# Load verified political layer data
report_path = Path('data/political/political_layer_report.json')
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
else:
    # Fallback: run analyzer to generate
    import political_layer_analyzer
    report = political_layer_analyzer.generate_report()

df = pd.DataFrame(report['member_table'])
df.head()

## Chart 1: Constituent Median Income by District

Shows the economic reality of the districts represented by members who control UI/SUI policy.

In [ ]:
house_df = df[df['chamber'] == 'House'].copy()
house_df = house_df.sort_values('constituent_median_income', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = house_df['state'].map({'MD': '#1f77b4', 'VA': '#ff7f0e', 'DC': '#2ca02c'})
bars = ax.barh(range(len(house_df)), house_df['constituent_median_income'], color=colors, alpha=0.8)

ax.set_yticks(range(len(house_df)))
ax.set_yticklabels([f"{r['name']} ({r['state']}-{r['district']})" for _, r in house_df.iterrows()], fontsize=9)
ax.set_xlabel('Median Household Income ($)', fontsize=12)
ax.set_title('Constituent Median Income by Congressional District (2022)
' + 
             'Districts Controlled by UI-Relevant Committee Members', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1f77b4', label='Maryland'),
                   Patch(facecolor='#ff7f0e', label='Virginia'),
                   Patch(facecolor='#2ca02c', label='DC')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## Chart 2: UI-Relevant Committee Members

Members serving on committees with jurisdiction over unemployment insurance, labor policy, and social safety net funding.

In [ ]:
ui_members = pd.DataFrame(report['ui_members_detail'])
ui_members['relevant_committees'] = ui_members['committees'].apply(
    lambda cs: [c for c in cs if any(kw in c.lower() for kw in ['ways and means', 'labor', 'education', 'workforce', 'finance', 'budget', 'appropriations'])]
)
ui_members['committee_count'] = ui_members['relevant_committees'].apply(len)

fig, ax = plt.subplots(figsize=(10, 6))
names = [f"{r['name']}
({r['state']})" for _, r in ui_members.iterrows()]
ax.barh(range(len(ui_members)), ui_members['committee_count'], color='crimson', alpha=0.8)
ax.set_yticks(range(len(ui_members)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Number of UI-Relevant Committee Assignments', fontsize=12)
ax.set_title('Members on UI-Relevant Committees (Ways & Means, Labor, Finance, Budget)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Chart 3: Party Composition of UI-Relevant Committees

Which party controls the committees that set SUI wage base caps and UI benefit levels?

In [ ]:
party_counts = ui_members['party'].value_counts()
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#0015BC' if p == 'Democratic' else '#FF0000' for p in party_counts.index]
ax.pie(party_counts.values, labels=party_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Party Composition of UI-Relevant Committee Members
' + 
             'From MD, VA, DC Delegations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Findings

1. **{len(ui_members)} members** from MD/VA/DC serve on UI-relevant committees (Ways & Means, Labor, Finance, Budget, Appropriations)
2. **Party composition**: {party_counts.to_dict()}
3. **Constituent income range**: ${house_df['constituent_median_income'].min():,} to ${house_df['constituent_median_income'].max():,}
4. **DC delegate** (Eleanor Holmes Norton) has no voting power on final passage but serves on Oversight and Transportation committees

## OpenSecrets Integration (Future)

When OpenSecrets API becomes accessible:
- Map member net worth ranges to constituent median income
- Identify top contributor industries (employer associations, business lobbies)
- Track STOCK Act transaction timing relative to UI legislation
- Expose the 'per-legislator portfolio' parallel to the 'per-employee injustice'